In [1]:
# Setup and imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add src to path
sys.path.append(str(Path('.').parent))

from src.dendritic_modeling.networks.layers.dendritic_conv2d import DendriteConv2d, DendriteAttention

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Create output directory
Path('demo_outputs').mkdir(exist_ok=True)

print("Setup complete!")


ModuleNotFoundError: No module named 'src'

In [ ]:
# Create a dendritic convolution layer
dendrite_conv = DendriteConv2d(
    in_channels=3,      # RGB input
    out_channels=16,    # 16 feature maps
    kernel_size=5,      # 5x5 kernels
    stride=1,
    padding=2,          # Same padding
    epsilon=1e-4
)

# Compare with standard convolution
standard_conv = nn.Conv2d(3, 16, kernel_size=5, stride=1, padding=2)

# Print layer info
print("Dendritic Conv Layer:")
print(f"  Input channels: {dendrite_conv.in_channels}")
print(f"  Output channels: {dendrite_conv.out_channels}")
print(f"  Kernel size: {dendrite_conv.kernel_size}")
print(f"  Excitatory weights shape: {dendrite_conv.weight_E.shape}")
print(f"  Inhibitory weights shape: {dendrite_conv.weight_I.shape}")

# Count parameters
dendrite_params = sum(p.numel() for p in dendrite_conv.parameters())
standard_params = sum(p.numel() for p in standard_conv.parameters())

print(f"\nParameter count:")
print(f"  Dendritic: {dendrite_params:,}")
print(f"  Standard: {standard_params:,}")
print(f"  Ratio: {dendrite_params/standard_params:.2f}x")


In [ ]:
# Create orientation-selective filters
def create_gabor_filter(size, theta, wavelength=4, sigma=1.5):
    """Create a Gabor filter for orientation detection."""
    x = np.arange(size) - size // 2
    y = np.arange(size) - size // 2
    X, Y = np.meshgrid(x, y)
    
    # Rotate coordinates
    X_rot = X * np.cos(theta) + Y * np.sin(theta)
    Y_rot = -X * np.sin(theta) + Y * np.cos(theta)
    
    # Gabor function
    gaussian = np.exp(-(X_rot**2 + Y_rot**2) / (2 * sigma**2))
    sinusoid = np.cos(2 * np.pi * X_rot / wavelength)
    
    gabor = gaussian * sinusoid
    return gabor

# Create a layer with 8 orientation-selective filters
n_orientations = 8
orientations = np.linspace(0, np.pi, n_orientations, endpoint=False)

orientation_layer = DendriteConv2d(
    in_channels=1,
    out_channels=n_orientations,
    kernel_size=9,
    padding=4,
    epsilon=1e-4
)

# Set weights to Gabor filters
with torch.no_grad():
    for i, theta in enumerate(orientations):
        # Create Gabor filter
        gabor = create_gabor_filter(9, theta)
        gabor_tensor = torch.tensor(gabor, dtype=torch.float32)
        
        # Set excitatory weights (positive part)
        excitatory = torch.clamp(gabor_tensor, min=0)
        orientation_layer.weight_E_raw.data[i, 0] = torch.log(excitatory + 1e-6)
        
        # Set inhibitory weights (negative part)
        inhibitory = torch.clamp(-gabor_tensor, min=0)
        orientation_layer.weight_I_raw.data[i, 0] = torch.log(inhibitory + 1e-6)

# Visualize the filters
fig, axes = plt.subplots(2, n_orientations, figsize=(16, 4))
for i in range(n_orientations):
    # Excitatory
    axes[0, i].imshow(orientation_layer.weight_E[i, 0].detach().numpy(), cmap='Reds')
    axes[0, i].set_title(f'{int(orientations[i] * 180/np.pi)}°')
    axes[0, i].axis('off')
    
    # Inhibitory
    axes[1, i].imshow(orientation_layer.weight_I[i, 0].detach().numpy(), cmap='Blues')
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Excitatory', rotation=90, size='large')
axes[1, 0].set_ylabel('Inhibitory', rotation=90, size='large')
plt.suptitle('Orientation-Selective E/I Filter Pairs', fontsize=14)
plt.tight_layout()
plt.savefig('demo_outputs/orientation_filters.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Create test images with oriented bars at different angles
def create_oriented_bar(size, angle, width=5):
    """Create an image with an oriented bar."""
    img = np.zeros((size, size))
    center = size // 2
    
    # Create coordinates
    y, x = np.ogrid[:size, :size]
    
    # Rotate coordinates
    x_rot = (x - center) * np.cos(angle) + (y - center) * np.sin(angle)
    
    # Create bar
    img[np.abs(x_rot) < width/2] = 1
    
    return img

# Test different orientations
test_angles = np.linspace(0, np.pi, 16, endpoint=False)
test_images = []

for angle in test_angles:
    img = create_oriented_bar(64, angle)
    test_images.append(img)

test_batch = torch.tensor(np.array(test_images), dtype=torch.float32).unsqueeze(1)

# Process through orientation layer
with torch.no_grad():
    responses = orientation_layer(test_batch)

# Plot results
fig, axes = plt.subplots(3, 8, figsize=(16, 6))

# Show first 8 test images
for i in range(8):
    axes[0, i].imshow(test_images[i], cmap='gray')
    axes[0, i].set_title(f'{int(test_angles[i] * 180/np.pi)}°')
    axes[0, i].axis('off')

# Show responses for each orientation detector
for i in range(8):
    # Get response map for this orientation detector across all test images
    orientation_responses = responses[:8, i].numpy()
    
    # Show response to best-matching stimulus
    best_idx = np.argmax([r.max() for r in orientation_responses])
    axes[1, i].imshow(orientation_responses[best_idx], cmap='hot')
    axes[1, i].set_title(f'Det {int(orientations[i] * 180/np.pi)}°')
    axes[1, i].axis('off')

# Plot tuning curves
for i in range(n_orientations):
    # Get maximum response for each test angle
    tuning_curve = [responses[j, i].max().item() for j in range(len(test_angles))]
    axes[2, i].plot(test_angles * 180/np.pi, tuning_curve, 'o-')
    axes[2, i].axvline(orientations[i] * 180/np.pi, color='red', linestyle='--', alpha=0.5)
    axes[2, i].set_xlabel('Stimulus angle (°)')
    if i == 0:
        axes[2, i].set_ylabel('Response')
    axes[2, i].set_ylim(0, max(tuning_curve) * 1.1)
    axes[2, i].grid(True, alpha=0.3)

axes[0, 0].set_ylabel('Stimulus', rotation=90, size='large')
axes[1, 0].set_ylabel('Response', rotation=90, size='large')
axes[2, 0].set_ylabel('Tuning', rotation=90, size='large')

plt.suptitle('Orientation Selectivity Test', fontsize=14)
plt.tight_layout()
plt.savefig('demo_outputs/orientation_selectivity.png', dpi=150, bbox_inches='tight')
plt.show()

# Calculate orientation selectivity index (OSI)
osi_values = []
for i in range(n_orientations):
    responses_i = [responses[j, i].max().item() for j in range(len(test_angles))]
    preferred = max(responses_i)
    orthogonal_idx = (np.argmax(responses_i) + len(test_angles)//2) % len(test_angles)
    orthogonal = responses_i[orthogonal_idx]
    
    osi = (preferred - orthogonal) / (preferred + orthogonal) if (preferred + orthogonal) > 0 else 0
    osi_values.append(osi)

print(f"Mean Orientation Selectivity Index: {np.mean(osi_values):.3f}")


In [ ]:
# Test contrast response
def test_contrast_response(layer, contrasts):
    """Test layer response to different contrast levels."""
    responses = []
    
    for contrast in contrasts:
        # Create high-contrast edge stimulus
        img = torch.zeros(1, 1, 32, 32)
        img[:, :, :, :16] = contrast
        img[:, :, :, 16:] = -contrast
        
        with torch.no_grad():
            output = layer(img)
            # Get maximum response
            max_response = output.abs().max().item()
            responses.append(max_response)
    
    return responses

# Create a simple edge detector
edge_detector = DendriteConv2d(1, 1, kernel_size=3, padding=1)

# Set up vertical edge detector
with torch.no_grad():
    edge_kernel = torch.tensor([[1, 0, -1], 
                               [1, 0, -1], 
                               [1, 0, -1]], dtype=torch.float32)
    
    # Excitatory: positive part
    edge_detector.weight_E_raw.data[0, 0] = torch.log(torch.clamp(edge_kernel, min=0) + 1e-6)
    # Inhibitory: constant inhibition
    edge_detector.weight_I_raw.data[0, 0] = torch.log(torch.ones_like(edge_kernel) * 0.3 + 1e-6)

# Test different contrast levels
contrasts = np.linspace(0, 2, 20)
dendrite_responses = test_contrast_response(edge_detector, contrasts)

# Compare with linear response (no divisive normalization)
linear_responses = []
for c in contrasts:
    # Simple linear filter response
    linear_response = c * 3  # 3 pixels in edge
    linear_responses.append(linear_response)

# Fit hyperbolic ratio function to dendritic response
from scipy.optimize import curve_fit

def hyperbolic_ratio(c, n, sigma):
    """Hyperbolic ratio function: R = c^n / (c^n + sigma^n)"""
    return c**n / (c**n + sigma**n)

# Fit the model
try:
    popt, _ = curve_fit(hyperbolic_ratio, contrasts[1:], dendrite_responses[1:], p0=[2, 0.5])
    fitted_response = hyperbolic_ratio(contrasts, *popt)
    fit_success = True
except:
    fit_success = False

# Plot contrast response functions
plt.figure(figsize=(8, 6))
plt.plot(contrasts, dendrite_responses, 'o-', label='Dendritic (Divisive Norm)', linewidth=2)
plt.plot(contrasts, linear_responses, '--', label='Linear (No Norm)', linewidth=2)
if fit_success:
    plt.plot(contrasts, fitted_response, ':', label=f'Hyperbolic fit (n={popt[0]:.2f}, σ={popt[1]:.2f})', linewidth=2)

plt.xlabel('Stimulus Contrast', fontsize=12)
plt.ylabel('Response', fontsize=12)
plt.title('Contrast Response Function: Divisive Normalization', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('demo_outputs/contrast_response.png', dpi=150, bbox_inches='tight')
plt.show()

# Calculate dynamic range improvement
linear_range = linear_responses[-1] - linear_responses[0]
dendrite_range = dendrite_responses[-1] - dendrite_responses[0]
compression_ratio = linear_range / dendrite_range

print(f"Dynamic range compression: {compression_ratio:.2f}x")
print(f"This allows the neuron to respond to a wider range of contrasts without saturation.")


In [ ]:
class DendriticCNN(nn.Module):
    """CNN using dendritic convolution layers."""
    
    def __init__(self, num_classes=10, use_dendritic=True):
        super(DendriticCNN, self).__init__()
        
        Conv = DendriteConv2d if use_dendritic else nn.Conv2d
        self.use_dendritic = use_dendritic
        
        # Layer 1: Edge detection (32 filters)
        self.conv1 = Conv(3, 32, kernel_size=5, stride=1, padding=2)
        self.pool1 = nn.MaxPool2d(2, 2)
        
        # Layer 2: Texture detection (64 filters)
        self.conv2 = Conv(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        
        # Layer 3: Object parts (128 filters)
        self.conv3 = Conv(64, 128, kernel_size=3, stride=1, padding=1)
        self.pool3 = nn.AdaptiveAvgPool2d((4, 4))
        
        # Classification head
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)
        
    def forward(self, x, return_features=False):
        features = []
        
        # Conv block 1
        x = self.conv1(x)
        x = F.relu(x)
        features.append(x.clone())
        x = self.pool1(x)
        
        # Conv block 2
        x = self.conv2(x)
        x = F.relu(x)
        features.append(x.clone())
        x = self.pool2(x)
        
        # Conv block 3
        x = self.conv3(x)
        x = F.relu(x)
        features.append(x.clone())
        x = self.pool3(x)
        
        # Classification
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        if return_features:
            return x, features
        return x

# Create both versions
dendritic_cnn = DendriticCNN(num_classes=10, use_dendritic=True)
standard_cnn = DendriticCNN(num_classes=10, use_dendritic=False)

# Compare architectures
print("Model Comparison:")
print("-" * 50)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

dendrite_params = count_parameters(dendritic_cnn)
standard_params = count_parameters(standard_cnn)

print(f"Dendritic CNN parameters: {dendrite_params:,}")
print(f"Standard CNN parameters: {standard_params:,}")
print(f"Parameter ratio: {dendrite_params/standard_params:.2f}x")

# Test on random input
test_input = torch.randn(1, 3, 32, 32)

with torch.no_grad():
    dendrite_out, dendrite_features = dendritic_cnn(test_input, return_features=True)
    standard_out, standard_features = standard_cnn(test_input, return_features=True)

print(f"\nOutput shape: {dendrite_out.shape}")
print(f"Feature shapes:")
for i, (d_feat, s_feat) in enumerate(zip(dendrite_features, standard_features)):
    print(f"  Layer {i+1}: {d_feat.shape}")


In [ ]:
# Visualize first layer filters
def visualize_layer1_filters(model, n_filters=16):
    """Visualize E/I filter pairs from first convolutional layer."""
    conv1 = model.conv1
    
    if isinstance(conv1, DendriteConv2d):
        E_filters = conv1.weight_E.detach()[:n_filters]
        I_filters = conv1.weight_I.detach()[:n_filters]
        
        fig, axes = plt.subplots(2, n_filters, figsize=(n_filters*1.5, 3))
        
        for i in range(n_filters):
            # Show RGB filters as grayscale (average across channels)
            e_filter = E_filters[i].mean(dim=0).numpy()
            i_filter = I_filters[i].mean(dim=0).numpy()
            
            # Normalize for visualization
            e_norm = (e_filter - e_filter.min()) / (e_filter.max() - e_filter.min() + 1e-8)
            i_norm = (i_filter - i_filter.min()) / (i_filter.max() - i_filter.min() + 1e-8)
            
            axes[0, i].imshow(e_norm, cmap='Reds', vmin=0, vmax=1)
            axes[0, i].axis('off')
            if i == 0:
                axes[0, i].set_title('E', fontsize=10)
            
            axes[1, i].imshow(i_norm, cmap='Blues', vmin=0, vmax=1)
            axes[1, i].axis('off')
            if i == 0:
                axes[1, i].set_title('I', fontsize=10)
        
        plt.suptitle('First Layer E/I Filter Pairs (Dendritic CNN)', fontsize=12)
    else:
        # Standard conv
        filters = conv1.weight.detach()[:n_filters]
        fig, axes = plt.subplots(1, n_filters, figsize=(n_filters*1.5, 1.5))
        
        for i in range(n_filters):
            filter_i = filters[i].mean(dim=0).numpy()
            filter_norm = (filter_i - filter_i.min()) / (filter_i.max() - filter_i.min() + 1e-8)
            axes[i].imshow(filter_norm, cmap='viridis')
            axes[i].axis('off')
        
        plt.suptitle('First Layer Filters (Standard CNN)', fontsize=12)
    
    plt.tight_layout()
    return fig

# Visualize filters
fig_dendrite = visualize_layer1_filters(dendritic_cnn)
plt.savefig('demo_outputs/dendritic_filters.png', dpi=150, bbox_inches='tight')
plt.show()

fig_standard = visualize_layer1_filters(standard_cnn)
plt.savefig('demo_outputs/standard_filters.png', dpi=150, bbox_inches='tight')
plt.show()

# Analyze E/I balance in first layer
if isinstance(dendritic_cnn.conv1, DendriteConv2d):
    conv1 = dendritic_cnn.conv1
    E_total = conv1.weight_E.sum(dim=(1,2,3)).detach().numpy()
    I_total = conv1.weight_I.sum(dim=(1,2,3)).detach().numpy()
    
    ei_ratio = E_total / (I_total + 1e-8)
    
    plt.figure(figsize=(8, 4))
    plt.subplot(1, 2, 1)
    plt.scatter(E_total, I_total, alpha=0.6)
    plt.xlabel('Total Excitation')
    plt.ylabel('Total Inhibition')
    plt.title('E/I Balance per Filter')
    plt.grid(True, alpha=0.3)
    
    # Add diagonal line for perfect balance
    max_val = max(E_total.max(), I_total.max())
    plt.plot([0, max_val], [0, max_val], 'k--', alpha=0.5, label='E=I')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.hist(ei_ratio, bins=20, alpha=0.7, edgecolor='black')
    plt.axvline(x=1.0, color='red', linestyle='--', label='E=I')
    plt.xlabel('E/I Ratio')
    plt.ylabel('Count')
    plt.title('Distribution of E/I Ratios')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('demo_outputs/ei_balance.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Mean E/I ratio: {ei_ratio.mean():.3f} ± {ei_ratio.std():.3f}")
    print(f"This suggests {'balanced' if 0.8 < ei_ratio.mean() < 1.2 else 'imbalanced'} excitation and inhibition")


In [ ]:
# Compare dendritic and standard attention
embed_dim = 128
num_heads = 4
seq_len = 16
batch_size = 2

# Create both attention mechanisms
dendrite_attn = DendriteAttention(embed_dim, num_heads)
standard_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

# Generate input sequence
x = torch.randn(batch_size, seq_len, embed_dim)

# Forward pass
with torch.no_grad():
    # Dendritic attention
    output_d, attn_weights_d = dendrite_attn(x, x, x)
    
    # Standard attention
    output_s, attn_weights_s = standard_attn(x, x, x, need_weights=True)

# Visualize attention patterns
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

# Show attention patterns for first 4 heads
for head in range(4):
    # Dendritic attention
    attn_d = attn_weights_d[0, head].detach().numpy()
    im1 = axes[0, head].imshow(attn_d, cmap='hot', aspect='auto', vmin=0)
    axes[0, head].set_title(f'Dendritic Head {head+1}')
    if head == 0:
        axes[0, head].set_ylabel('Queries')
    axes[0, head].set_xlabel('Keys')
    
    # Standard attention  
    attn_s = attn_weights_s[0, :, :].detach().numpy()  # Standard returns averaged heads
    im2 = axes[1, head].imshow(attn_s, cmap='hot', aspect='auto', vmin=0)
    axes[1, head].set_title(f'Standard (avg)')
    if head == 0:
        axes[1, head].set_ylabel('Queries')
    axes[1, head].set_xlabel('Keys')

# Add colorbars
plt.colorbar(im1, ax=axes[0, :], shrink=0.8, label='Attention Weight')
plt.colorbar(im2, ax=axes[1, :], shrink=0.8, label='Attention Weight')

plt.suptitle('Attention Pattern Comparison', fontsize=14)
plt.tight_layout()
plt.savefig('demo_outputs/attention_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

# Analyze attention properties
# 1. Sparsity
sparsity_d = (attn_weights_d > 0.1).float().mean().item()
sparsity_s = (attn_weights_s > 0.1).float().mean().item()

# 2. Energy conservation (row sums)
row_sums_d = attn_weights_d.sum(dim=-1)
row_sums_s = attn_weights_s.sum(dim=-1)

# 3. Entropy
def attention_entropy(attn):
    """Calculate entropy of attention distribution."""
    # Add small epsilon to avoid log(0)
    attn_safe = attn + 1e-10
    entropy = -(attn_safe * torch.log(attn_safe)).sum(dim=-1)
    return entropy.mean().item()

entropy_d = attention_entropy(attn_weights_d)
entropy_s = attention_entropy(attn_weights_s)

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

# Sparsity comparison
axes[0].bar(['Dendritic', 'Standard'], [sparsity_d, sparsity_s], 
            color=['coral', 'skyblue'], alpha=0.7)
axes[0].set_ylabel('Fraction > 0.1')
axes[0].set_title('Attention Sparsity')
axes[0].set_ylim(0, 1)

# Row sum distribution
axes[1].hist(row_sums_d.flatten().numpy(), bins=30, alpha=0.7, 
             label='Dendritic', color='coral', density=True)
axes[1].hist(row_sums_s.flatten().numpy(), bins=30, alpha=0.7, 
             label='Standard', color='skyblue', density=True)
axes[1].axvline(x=1.0, color='black', linestyle='--', label='Sum=1')
axes[1].set_xlabel('Row Sum')
axes[1].set_ylabel('Density')
axes[1].set_title('Attention Weight Sums')
axes[1].legend()

# Entropy comparison
axes[2].bar(['Dendritic', 'Standard'], [entropy_d, entropy_s], 
            color=['coral', 'skyblue'], alpha=0.7)
axes[2].set_ylabel('Entropy (nats)')
axes[2].set_title('Attention Entropy')

plt.suptitle('Attention Mechanism Properties', fontsize=14)
plt.tight_layout()
plt.savefig('demo_outputs/attention_properties.png', dpi=150, bbox_inches='tight')
plt.show()

print("Attention Analysis:")
print(f"  Sparsity - Dendritic: {sparsity_d:.3f}, Standard: {sparsity_s:.3f}")
print(f"  Mean row sum - Dendritic: {row_sums_d.mean():.3f}, Standard: {row_sums_s.mean():.3f}")
print(f"  Entropy - Dendritic: {entropy_d:.3f}, Standard: {entropy_s:.3f}")
print(f"\nDendritic attention is {'sparser' if sparsity_d < sparsity_s else 'denser'} and has {'lower' if entropy_d < entropy_s else 'higher'} entropy.")
